In [1]:
# Step 1: Imports
import torch
from torch import nn
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split

In [2]:
# Step 2: Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cpu


In [3]:
# Step 3: Dataset (moons with higher noise so training is non-trivial)
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, y_train, X_test, y_test = X_train.to(device), y_train.to(device), X_test.to(device), y_test.to(device)

In [4]:
# Step 4: Model definition
class CircleModel(nn.Module):
    def __init__(self, hidden_units=10, activation=nn.ReLU()):
        super().__init__()
        self.layer_1 = nn.Linear(2, hidden_units)
        self.layer_2 = nn.Linear(hidden_units, hidden_units)
        self.layer_3 = nn.Linear(hidden_units, 1)
        self.activation = activation

    def forward(self, x):
        return self.layer_3(self.activation(self.layer_2(self.activation(self.layer_1(x)))))


In [5]:

# Step 5: Accuracy function
def accuracy_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item()
    return (correct / len(y_true)) * 100


In [6]:

# Step 6: Training function
def train_model(model, X_train, y_train, X_test, y_test, epochs=500, lr=0.05, optimizer_cls=torch.optim.SGD):
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = optimizer_cls(model.parameters(), lr=lr)

    train_losses, test_losses = [], []
    train_accs, test_accs = [], []

    for epoch in range(epochs):
        # Training
        model.train()
        y_logits = model(X_train).squeeze()
        y_pred = torch.round(torch.sigmoid(y_logits))
        loss = loss_fn(y_logits, y_train)
        acc = accuracy_fn(y_train, y_pred)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Testing
        model.eval()
        with torch.inference_mode():
            test_logits = model(X_test).squeeze()
            test_preds = torch.round(torch.sigmoid(test_logits))
            test_loss = loss_fn(test_logits, y_test)
            test_acc = accuracy_fn(y_test, test_preds)

        train_losses.append(loss.item())
        test_losses.append(test_loss.item())
        train_accs.append(acc)
        test_accs.append(test_acc)

        if (epoch+1) % 100 == 0:
            print(f"Epoch {epoch+1} | Train Loss {loss:.4f} | Train Acc {acc:.2f}% | Test Loss {test_loss:.4f} | Test Acc {test_acc:.2f}%")

    return {
        "final_train_acc": train_accs[-1],
        "final_test_acc": test_accs[-1],
        "train_losses": train_losses,
        "test_losses": test_losses,
        "train_accs": train_accs,
        "test_accs": test_accs
    }


In [7]:

# Step 7: Run base experiment
model = CircleModel(hidden_units=10, activation=nn.ReLU()).to(device)
result_base = train_model(model, X_train, y_train, X_test, y_test, epochs=2000, lr=0.05, optimizer_cls=torch.optim.SGD)


Epoch 100 | Train Loss 0.6753 | Train Acc 50.00% | Test Loss 0.6754 | Test Acc 50.00%
Epoch 200 | Train Loss 0.5671 | Train Acc 80.75% | Test Loss 0.5779 | Test Acc 77.00%
Epoch 300 | Train Loss 0.4290 | Train Acc 84.75% | Test Loss 0.4441 | Test Acc 81.50%
Epoch 400 | Train Loss 0.3446 | Train Acc 86.38% | Test Loss 0.3523 | Test Acc 84.50%
Epoch 500 | Train Loss 0.3049 | Train Acc 86.88% | Test Loss 0.3068 | Test Acc 85.50%
Epoch 600 | Train Loss 0.2893 | Train Acc 87.62% | Test Loss 0.2877 | Test Acc 87.00%
Epoch 700 | Train Loss 0.2814 | Train Acc 88.00% | Test Loss 0.2783 | Test Acc 87.00%
Epoch 800 | Train Loss 0.2758 | Train Acc 88.00% | Test Loss 0.2722 | Test Acc 87.00%
Epoch 900 | Train Loss 0.2710 | Train Acc 88.25% | Test Loss 0.2669 | Test Acc 87.50%
Epoch 1000 | Train Loss 0.2662 | Train Acc 88.62% | Test Loss 0.2615 | Test Acc 88.00%
Epoch 1100 | Train Loss 0.2610 | Train Acc 88.62% | Test Loss 0.2557 | Test Acc 88.00%
Epoch 1200 | Train Loss 0.2552 | Train Acc 88.38% | 

In [8]:

# Step 8: Save summary.txt
log_lines = []
header = f"{'Experiment':<35}{'Train Acc %':<15}{'Test Acc %':<15}"
separator = "-" * 65
log_lines.append(header)
log_lines.append(separator)
line = f"{'Base model (moons)':<35}{result_base['final_train_acc']:<15.2f}{result_base['final_test_acc']:<15.2f}"
log_lines.append(line)

with open("results_summary.txt", "w") as f:
    f.write("\n".join(log_lines))

print("\nTraining complete. Results saved to results_summary.txt")



Training complete. Results saved to results_summary.txt
